In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

TODO:

- Try class for `Card` later? `namedtuple` works for now.
- Try `IntEnum` for Poker hand rankings. Currently, we use integer values from 14 to 1 to indicate the poker hand rankings. Using `IntEnum` may make it more readable. Read more about enum: https://docs.python.org/3/library/enum.html
- Try passing `Round` to `Player` when taking action to update `stage_bet`.
- Player should not be able to check if `stage_bet != player.bet`.
- Add possible actions for players.
- **Odd chips vanish** — `self.pot // len(winners)` drops the remainder
- Implement the rule: If everyone folds except for one player, that player wins

In [ ]:
#| export
from fastcore.utils import *
from collections import namedtuple
from collections import Counter
import random
from enum import Enum, auto
from dataclasses import dataclass, field

# core

> Fill in a module description here

In this dialog, I want to go through the process of writing a poker program myself but following the approach I learned in the cs212 class. I went through the lesson in `pytude/cs212/lesson_1` dialog. Read the dialog to guide me when I ask for help without giving me answers directly. It's been some time since I went over this lesson, so I mostly likely forgot about all the details of implementations. My goal is not to copy everything from the lesson exactly, but follow the steps: understanding the problem, specifying, concept inventory, etc. that I can apply to solving any problem. To challenge myself, I will write texas hold'em poker.

Here's the process Norvig taught:

**1. Understand** — Start with a vague understanding and refine it. Make an inventory of all the concepts you'll need to deal with (data types, operations, etc.).

**2. Specify** — Define *what* needs to happen: inputs, outputs, what each function should accomplish. Think of it as a contract — not the implementation details, just the "what."

**3. Design** — Figure out *how* to make it happen — the actual algorithms, code structure, and implementation.

Along the way, key principles:
- **Write tests early** (even before implementation), including **extreme values**
- **Reuse existing tools** when possible
- **Refactor** for elegance (DRY, clarity, simplicity, generality)
- Be aware of the **four dimensions**: correctness, efficiency, features, elegance — and make conscious tradeoffs

## Understanding the problem.

Rules in the poker:

Poker is a card game with 52 cards, and each card has a suit and a rank. A suit consists of spades, clubs, hearts, and diamonds. A rank consists of numbers from ace, 2 to 10, jack, queen, and king. A hand has two cards, and community cards have five cards. They are combined to make different combinations used for comparisons among hands. At the start of the round, each player gets two cards face down and three cards in the community. After each betting round, one additional card is dealt to the community until there are five cards. Each hand from the players is compared, and winner takes all the bets unless there is a tie, and they split. To compare the hands, poker ranking is used. At each betting round, players can call, check, raise, or fold. If everyone folds except for one player, that player wins. If a player raises, others have to match the bet (call) or raise, or fold. After everyone checks, next round starts. 

Poker hands:
- royal flush
- straight flush
- four of a kind
- full house
- flush
- straight
- three of a kind
- two pair
- one pair
- high card

Concept inventory

Core concepts:
  - `Rank`: 2 to 10, jack, queen, king, and ace
  - `suit`: diamonds, hearts, clubs, spades.
  - `Card`: with rank and suit.
  - `CommunityCards`: Three to five cards shared by all the players.
  - `Hand`: 2 hole cards.
  - `HandRank`: Combining `Hand` with `CommunityCards`, hand rankings are compared among the players. From 7 cards, best 5 cards are used.
  - `Stage`: A stage is where each player takes actions such as call, fold, raise, and check. It ends when everyone folds except for those who called.
  - `Round`: A round consists of four stages (pre-flop, after flop, turn, and river). 
  - `Stage_bet`: money to match each round for each player and becomes the prize pool.
  - `Bet`: Money each player put in this stage.
  - `Balance`: The amount of moeny each player possesses so they can use for betting.
  - `Player state`: Useful player state such as balance, hand, whether playing the round or not, and bet.
  - `Deck`: 52 cards that get shuffled.
  - `Pot`: prize money that gets distributed to winners at the end of each round.

Add-ons:
  - `Blinds`: small and big blind. Small is immediately to the left of the dealer, and big blind is after. Big blind is twice the small blind amount.
  - `Dealer`: rotates clockwise each hand.
  - `display`: text based or graphical display.
  - `GameObserver/AI advisor`: sees the public game state and hole cards. Answers questions on request.

Operations

- Shuffle the deck
- Deal cards
- Perform player actions: call, fold, check, raise
- Evaluate a hand rank
- Compare hand ranks
- Distribute the pot to winners
- Start/end rounds
- Rotate dealer
- Draw community cards

**Specification**

Specifying is the step where you define **what** your program should do, without worrying about *how* it does it. It's like writing a contract:

- What are the **inputs**?
- What are the **outputs**?
- What **behavior** should the program have?

--- 

Card & Deck Operations
- `shuffle(deck) -> None`: Shuffle the deck.
- `deal_hole(deck, players) -> None`: Deal 2 cards to each player.
- `deal_community(deck, community_cards, n) -> None`: Deal n cards from deck to community cards.

Hand Evaluation
  - `evaluate_hand(hand, community_cards) -> hand_rank`: From 7 cards, find the best 5-card hand and return its rank.
  - `compare_ranks(hand_ranks) -> tuple`: Given a set of hand ranks, return the best one.

Player Actions
  - `perform_action(player, round) -> None`: Prompt player for an action, then call the appropriate function.
  - `call(player, round) -> None`: Player matches the current bet. Round state updated.
  - `fold(player, round) -> None`: Player folds. Round state updated.
  - `check(player, round) -> None`: Player checks. Round state updated.
  - `raise(player, round, amount) -> None`: Player raises by amount. Round state updated.

Round Management
  - `betting_round(round) -> None`: Manage one round of player actions (pre-flop, flop, turn, or river). Ends when all active players have called or checked.
  - `start_round(round) -> None`: Orchestrate the whole hand: deal hole cards → blinds → betting round → flop → betting round → turn → betting round → river → betting round → showdown → distribute.
  - `distribute(pot, winners) -> None`: Distribute the pot to the winner(s).

State
- round: tracks players with their bets, pot, community cards, and current betting state.
- player: has balance, current bet, hand, and whether folded or active.

**Design**

Design is where you figure out **how** to make it happen. The spec said *what* your program does; design says *how* it does it.

This is where you make decisions like:

- **Data representation** — How do you represent a card? A tuple `(rank, suit)`? A string `"AS"`? A custom class?
- **Code structure** — How do you organize the functions? What goes in a class vs standalone functions?
- **Algorithms** — How do you evaluate the best 5-card hand from 7? How do you determine when a betting round is over?

It's still not about writing the final code — it's about making the key implementation choices before you start coding.


`Card`:
- In the beginning, I tried using a string "♠2". This was a very simple and easy way to represent a card. To grab a rank or a suit from a card, `c1`, I can do `c1[0]` or `c1[1:]`. To make it more readable, I defined a function for these operations. Then I thought it was more convenient to use a dot notation. Rather than using a class, I tried using `namedtuple` from `collections`. This was a very light weight.

class `Player` with attributes 
- balance: int
- bet: int
- playing: bool
- hand: List[Card]
methods: 
- take_action(round)
- call(round)
- check(round)
- fold(round)
- raise(round, amount). 

All the methods need to pass `round` and update it. Therefore, I might get a `round` attribute. But this leads to circular dependency, which tightly couples these together. It becomes harder to test.

`Round` class attributes are 
- players: List[Player]
- dealer: int
- deck: Deck
- stage: Enum(pre-flop, flop, turn, river)
- pot: int
- community: CommunityCard

methods:
- betting_round()
- start_round()
- distribute()

**Architecture**

`Game` is the top-level mediator. It owns `players` and the current `round`. Neither `Player` nor `Round` reference each other directly — `Game` coordinates between them.

**State ownership**

`Round` is the single owner of all mutable state: player balances, bets, pot, community cards, dealer position, and stage. All state updates flow through `Round`.

**Functional data operations**

`Deck` and `Card` operations are pure — they return new values rather than mutating:

- `shuffle() -> Deck` — returns a new shuffled deck
- `deal(n) -> (cards, Deck)` — returns dealt cards and a new deck with remaining cards

**Pure logic functions**

Hand evaluation and bet calculations:

- `evaluate_hand(hand, community) -> HandRank` — given 7 cards, returns the best 5-card rank
- `compare_ranks(ranks) -> tuple` — returns the winner rank

**Round methods**

`Round` applies results from pure functions to update state:

- `betting_round()` — manages one round of actions
- `start_round()` — orchestrates the full hand
- `distribute(winners)` — distributes pot to winners

**Player actions**

Players receive only the data they need (current bet, pot, etc.) rather than a full `Round` reference. Actions return *what* the new state should be, and `Round` applies it.

How to compare hands? Let's use a tuple with first entry representing the `hand_rank` from royal flush to hgih card with 10 to 1. Then later entries for tie breakers.

Poker hands:
- royal flush (10)
- straight flush (9, highest_value)
- four of a kind (8, highest_value, kicker)
- full house (7, triple, pair)
- flush (6, five kickers)
- straight (5, highest_value)
- three of a kind (4, triple_value, two kickers)
- two pair (3, high_pair, low_pair, kicker)
- one pair (2, pair_value, three kickers)
- high card (1, highest, second highest, third, fourth, fifth)

## Card

There are many ways to design `Card`. One of the simplest way is to use string.

I use emoji for suits and string of alphanumerics for ranks.

In [ ]:
#| export
suits = '♠♥♦♣'
suits

'♠♥♦♣'

In [ ]:
#| export
ranks = '2,3,4,5,6,7,8,9,10,J,Q,K,A'.split(',')
ranks

['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']

Rather than using a string for cards and using separate functions to access suits and ranks, I thought it would be better to use the `.` (dot) notation. Rather than using a `class`, I wanted to try using `namedtuple`, which allows me to use attribute access and immutability. The ranks and suits from each card should not change. It is more lightweight than `class`.

In [ ]:
#| export
Card = namedtuple('Card', ['suit', 'rank'])
Card

__main__.Card

In [ ]:
c1 = Card('♠', '10')
c1.suit, c1.rank

('♠', '10')

In [ ]:
c1

Card(suit='♠', rank='10')

I want the card representation to be more concise, so I patch it.

In [ ]:
#| export
@patch
def __repr__(self: Card): return self.suit + self.rank

In [ ]:
c2 = Card('♠', 'K')
c2

♠K

Currently, we cannot compare the ranks directly as we are using string comparisons lexicographically. 

In [ ]:
c1.rank > c2.rank, '10' > 'K'

(False, False)

It is convenient to sort hands by rank values when evaluating hands. 

Using `rank_values` mapping, we can convert the string ranks into integers. From 2 to 10, their values are 2 to 10 respectively. Then, from 'J' to 'A', 11 to 14.

In [ ]:
#| export
rank_values = {r:i for i,r in enumerate(ranks, start=2)}
rank_values

{'2': 2,
 '3': 3,
 '4': 4,
 '5': 5,
 '6': 6,
 '7': 7,
 '8': 8,
 '9': 9,
 '10': 10,
 'J': 11,
 'Q': 12,
 'K': 13,
 'A': 14}

Although I had `sort_cards` function that sorts cards by their `rank_values` from the highest, it was more convenient to patch `__lt__` and `__eq__` and just use default `sorted` function. When comparing cards, we only care about their ranks, not ranks. 

In [ ]:
#| export
@patch
def __lt__(self: Card, other: Card): return rank_values[self.rank] < rank_values[other.rank]

In [ ]:
#| export
@patch
def __eq__(self: Card, other: Card): return rank_values[self.rank] == rank_values[other.rank]

In [ ]:
deck = [Card(s,r) for s in suits for r in ranks]
sorted(deck,reverse=True)[:5]

[♠A, ♥A, ♦A, ♣A, ♠K]

## Hand

- `evaluate_hand(cards) -> HandRank` — given 7 cards, returns the best 5-card `HandRank`
- `compare_ranks(ranks) -> tuple` — returns the winner rank

A hand has two cards. Community cards have five cards. 

Poker hands:
- royal flush (10)
- straight flush (9, highest_value)
- four of a kind (8, highest_value, kicker)
- full house (7, triple, pair)
- flush (6, five kickers)
- straight (5, highest_value)
- three of a kind (4, triple_value, two kickers)
- two pair (3, high_pair, low_pair, kicker)
- one pair (2, pair_value, three kickers)
- high card (1, highest, second highest, third, fourth, fifth)

In [ ]:
c1, c2

(♠10, ♠K)

In [ ]:
c1.rank

'10'

### Sample hands

These are some example hands.

In [ ]:
cards_highcard = [Card(suits[0], ranks[0]), Card(suits[1], ranks[2]), Card(suits[2], ranks[4]),
    Card(suits[3], ranks[6]), Card(suits[0], ranks[8]), Card(suits[1], ranks[10]), Card(suits[2], ranks[12])]
cards_highcard

[♠2, ♥4, ♦6, ♣8, ♠10, ♥Q, ♦A]

In [ ]:
cards_2kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[0], ranks[3]), Card(suits[1], ranks[5]), 
    Card(suits[2], ranks[7]), Card(suits[3], ranks[9]), Card(suits[0], ranks[11])]
cards_2kind

[♠2, ♥2, ♠5, ♥7, ♦9, ♣J, ♠K]

In [ ]:
cards_3kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]), Card(suits[0], ranks[3]), 
    Card(suits[1], ranks[5]), Card(suits[2], ranks[7]), Card(suits[3], ranks[9])]
cards_3kind

[♠2, ♥2, ♦2, ♠5, ♥7, ♦9, ♣J]

In [ ]:
cards_4kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]), Card(suits[3], ranks[0]),
    Card(suits[0], ranks[4]), Card(suits[1], ranks[7]), Card(suits[2], ranks[10])]
cards_4kind

[♠2, ♥2, ♦2, ♣2, ♠6, ♥9, ♦Q]

In [ ]:
cards_2pair = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[1]), Card(suits[3], ranks[1]),
    Card(suits[0], ranks[4]), Card(suits[1], ranks[6]), Card(suits[2], ranks[8])]
cards_2pair

[♠2, ♥2, ♦3, ♣3, ♠6, ♥8, ♦10]

In [ ]:
cards_fullhouse = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[0], ranks[1]), Card(suits[2], ranks[1]), 
    Card(suits[3], ranks[1]), Card(suits[1], ranks[6]), Card(suits[2], ranks[8])]
cards_fullhouse

[♠2, ♥2, ♠3, ♦3, ♣3, ♥8, ♦10]

In [ ]:
cards_straight = [Card(suits[0], ranks[1]), Card(suits[1], ranks[2]), Card(suits[2], ranks[3]),
    Card(suits[3], ranks[4]), Card(suits[0], ranks[5]), Card(suits[1], ranks[8]), Card(suits[2], ranks[10])]
cards_straight

[♠3, ♥4, ♦5, ♣6, ♠7, ♥10, ♦Q]

In [ ]:
cards_flush = [Card(suits[0], ranks[0]), Card(suits[0], ranks[2]), Card(suits[0], ranks[4]),
    Card(suits[0], ranks[6]), Card(suits[0], ranks[8]), Card(suits[1], ranks[3]), Card(suits[2], ranks[7])]
cards_flush

[♠2, ♠4, ♠6, ♠8, ♠10, ♥5, ♦9]

In [ ]:
cards_straightflush = [Card(suits[0], ranks[1]), Card(suits[0], ranks[2]), Card(suits[0], ranks[3]),
    Card(suits[0], ranks[4]), Card(suits[0], ranks[5]), Card(suits[1], ranks[8]), Card(suits[2], ranks[10])]
cards_straightflush

[♠3, ♠4, ♠5, ♠6, ♠7, ♥10, ♦Q]

In [ ]:
cards_royalflush = [Card(suits[0], ranks[12]), Card(suits[0], ranks[11]), Card(suits[0], ranks[10]), 
    Card(suits[0], ranks[9]), Card(suits[0], ranks[8]), Card(suits[1], ranks[2]), Card(suits[2], ranks[4])]
cards_royalflush

[♠A, ♠K, ♠Q, ♠J, ♠10, ♥4, ♦6]

### Ranks and counts

By getting the counts from the cards, we can figure out `four of a kind`, `full house`, `three of a kind`, `two pair`, and `one pair`. One thing we have to be careful of is because we have seven cards to evaluate, `full house` might have two `three of kind`s, rather than a `three of kind` and a `pair`.

Here is `get_ranks_counts`, which takes a list of cards and returns a tuple of ranks and counts. The output is sorted by the rank counts and `rank_values` for tie breakers. Rank counts are useful for finding what kind of poker ranking hand it is, and ranks are used for the main hand components and kickers.

In [ ]:
#| export
def get_ranks_counts(cards: list[Card]) -> tuple[tuple[int, ...], tuple[int, ...]]:
    """Return ((ranks...), (counts...)) sorted by count then by rank value from cards"""
    items = Counter([rank_values[c.rank] for c in cards]).items()
    return tuple(zip(*sorted(items, key=lambda o: (o[1], o[0]), reverse=True)))

In [ ]:
get_ranks_counts(cards_fullhouse)

((3, 2, 10, 8), (3, 2, 1, 1))

In [ ]:
def test_get_ranks_counts():
    assert get_ranks_counts(cards_2kind) == ((2, 13, 11, 9, 7, 5), (2, 1, 1, 1, 1, 1))
    assert get_ranks_counts(cards_3kind) == ((2, 11, 9, 7, 5), (3, 1, 1, 1, 1))
    assert get_ranks_counts(cards_4kind) == ((2, 12, 9, 6), (4, 1, 1, 1))
    assert get_ranks_counts(cards_2pair) == ((3, 2, 10, 8, 6), (2, 2, 1, 1, 1))
    assert get_ranks_counts(cards_fullhouse) == ((3, 2, 10, 8), (3, 2, 1, 1))
    assert get_ranks_counts(cards_highcard) == ((14, 12, 10, 8, 6, 4, 2), (1, 1, 1, 1, 1, 1, 1))
    assert get_ranks_counts(cards_straight) == ((12, 10, 7, 6, 5, 4, 3), (1, 1, 1, 1, 1, 1, 1))

    # Edge case: two three-of-a-kinds (e.g. 222 333 K) — full house should pick higher triple
    cards_2trips = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]),
                    Card(suits[0], ranks[1]), Card(suits[1], ranks[1]), Card(suits[2], ranks[1]),
                    Card(suits[3], ranks[11])]
    assert get_ranks_counts(cards_2trips) == ((3, 2, 13), (3, 3, 1))

test_get_ranks_counts()

### flush

As long as the suit count is 5 or more, it is flush because we have seven cards total.

In [ ]:
#| export
def is_flush(cards: list[Card]) -> list:
    "If cards have flush, return the `rank_values` from the cards in sorted order. If not, return []"
    suit, count = Counter(c.suit for c in cards).most_common()[0]
    if count >= 5:
        return sorted([rank_values[c.rank] for c in cards if c.suit == suit][:5], reverse=True)
    return []

In [ ]:
def test_is_flush():
    # Flush: 5 spades, two side cards
    assert is_flush(cards_flush) == [10, 8, 6, 4, 2]

    # Straight flush
    assert is_flush(cards_straightflush) == [7, 6, 5, 4, 3]

    # Royal flush
    assert is_flush(cards_royalflush) == [14, 13, 12, 11, 10]

    # No flush: suits too spread out
    assert is_flush(cards_fullhouse) == []
    assert is_flush(cards_highcard) == []
    assert is_flush(cards_4kind) == []

    # Edge: all 7 cards same suit — returns top 5
    all_spades = [Card('♠', r) for r in 'A,K,Q,J,10,9,8'.split(',')]
    assert is_flush(all_spades) == [14, 13, 12, 11, 10]

    # Edge: exactly 5 same suit
    five_hearts = [Card('♥', r) for r in '2,3,4,5,6'.split(',')] + [Card('♠', 'A'), Card('♦', 'K')]
    assert is_flush(five_hearts) == [6, 5, 4, 3, 2]

test_is_flush()

### straight

Finding straight is trickier because Ace can serve as 1 or 14. Also, there are seven cards, but only five are used. 

In [ ]:
#| export
def is_straight(cards: list[Card]) -> int:
    "Return the highest value of the straight if cards contain 5 consecutive ranks, else 0"
    vals = sorted([rank_values[c.rank] for c in cards], reverse=True)
    if vals[0] == 14: 
        vals.append(1)
    for i in range(len(vals) - 4):
        if vals[i] - vals[i+4] == 4 and len(set(vals[i:i+5])) == 5:
            return vals[i]
    return 0

In [ ]:
def test_is_straight():
    # Regular straight: 3-4-5-6-7
    assert is_straight(cards_straight) == 7
    
    # Ace-high straight (10-J-Q-K-A)
    ace_high = [Card('♠','10'), Card('♥','J'), Card('♦','Q'), Card('♣','K'), Card('♠','A'),
                Card('♥','3'), Card('♦','6')]
    assert is_straight(ace_high) == 14
    
    # Wheel: A-2-3-4-5 (ace low)
    wheel = [Card('♠','A'), Card('♥','2'), Card('♦','3'), Card('♣','4'), Card('♠','5'),
             Card('♥','9'), Card('♦','J')]
    assert is_straight(wheel) == 5
    
    # No straight — gapped
    assert is_straight(cards_2pair) == 0
    
    # No straight — bunch of duplicates that run together in rank
    dupes = [Card('♠','5'), Card('♥','5'), Card('♦','6'), Card('♣','6'), Card('♠','7'),
             Card('♥','7'), Card('♦','8')]
    assert is_straight(dupes) == 0

test_is_straight()

### evaluate_hand

Now that we can detect flush and straight, we can update `evaluate_hand`.

In [ ]:
#| export
def evaluate_hand(cards: list[Card]) -> tuple:
    """Return hand rank tuple for the best 5-card hand from `cards`.
    Hands ranked from 1 (high card) to 10 (royal flush). Each rank tuple
    starts with the hand category, followed by tiebreaker values (kickers
    or relevant ranks).
    """
    ranks, counts = get_ranks_counts(cards)
    straight_val = is_straight(cards)
    flush_vals = is_flush(cards)
    if straight_val == 14: # royal flush
        return (10,)
    elif straight_val != 0 and flush_vals != []: # straight flush
        return (9, straight_val)
    elif counts[0] == 4: # four of a kind
        return (8, ranks[0], ranks[1])
    elif counts[0] == 3 and counts[1] >= 2: # full house
        return (7, ranks[0], ranks[1])
    elif flush_vals != []: # flush
        return (6, *flush_vals)
    elif straight_val != 0: # straight
        return (5, straight_val)
    elif counts[0] == 3: # three of a kind
        return (4, ranks[0], ranks[1], ranks[2])
    elif counts[0] == 2 and counts[1] == 2: # two pair
        return (3, ranks[0], ranks[1], ranks[2])
    elif counts[0] == 2: # one pair
        return (2, ranks[0], ranks[1], ranks[2], ranks[3])
    else: # high card
        return (1, *ranks)

In [ ]:
def test_evaluate_hand():
    # Royal flush: A♠ K♠ Q♠ J♠ 10♠ → (10,)
    assert evaluate_hand(cards_royalflush) == (10,)

    # Straight flush: 3♠ 4♠ 5♠ 6♠ 7♠ → (9, 7)
    assert evaluate_hand(cards_straightflush) == (9, 7)

    # Four of a kind: 2222 + Q 9 6 → (8, 2, 12)
    assert evaluate_hand(cards_4kind) == (8, 2, 12)

    # Full house: 333 22 + 10 8 → (7, 3, 2)
    assert evaluate_hand(cards_fullhouse) == (7, 3, 2)

    # Flush: 5 spades → (6, 10, 8, 6, 4, 2)
    assert evaluate_hand(cards_flush) == (6, 10, 8, 6, 4, 2)

    # Straight: 3-4-5-6-7 → (5, 7)
    assert evaluate_hand(cards_straight) == (5, 7)

    # Three of a kind: 222 + J 9 7 5 → (4, 2, 11, 9)
    assert evaluate_hand(cards_3kind) == (4, 2, 11, 9)

    # Two pair: 33 22 + 10 8 6 → (3, 3, 2, 10)
    assert evaluate_hand(cards_2pair) == (3, 3, 2, 10)

    # One pair: 22 + K J 9 7 5 → (2, 2, 13, 11, 9)
    assert evaluate_hand(cards_2kind) == (2, 2, 13, 11, 9)

    # High card: A Q 10 8 6 4 2 → (1, 14, 12, 10, 8, 6, 4, 2)
    assert evaluate_hand(cards_highcard) == (1, 14, 12, 10, 8, 6, 4, 2)

test_evaluate_hand()

### compare_hands

`compare_hands(hands: List[List[Card]]) -> tuple` — returns the ranks for the winner(s)

Winners are those who have the winning ranks

In [ ]:
#| export
def compare_hands(hands: list[list[Card]]) -> tuple:
    """Return the winning hand rank among `hands`."""
    return sorted([evaluate_hand(h) for h in hands], reverse=True)[0]

In [ ]:
evaluate_hand(cards_flush)

(6, 10, 8, 6, 4, 2)

In [ ]:
def test_compare_hands():
    # Clear winner: flush beats one pair
    assert compare_hands([cards_2kind, cards_flush]) == (6, 10, 8, 6, 4, 2)
    
    # Tie: two identical hands → both should win
    assert compare_hands([cards_flush, cards_flush]) == (6, 10, 8, 6, 4, 2)
    
    # Three hands, two tie for best
    assert compare_hands([cards_2kind, cards_flush, cards_royalflush]) == (10,)
    
    # Reverse order shouldn't matter
    assert compare_hands([cards_flush, cards_2kind]) == (6, 10, 8, 6, 4, 2)

test_compare_hands()

## Deck

In my original plan, I thought of writing a `CardCollection` class as an abstract class and create `Hand`, `Deck`, and `CommunityCards` by inheriting from it as they all consist of cards. However, they don't really have methods that share. To keep things simple, I decided to start implementing `Deck` as a list of `Card`s and see how it feels.

A deck consists of 52 cards. From that, we need to shuffle and withdraw cards so the dealer can deal cards.

In [ ]:
#| export
def shuffle_deck(deck: list[Card]) -> list[Card]:
    "Shuffle `deck` in place and return it"
    random.shuffle(deck)
    return deck

In [ ]:
random.seed(42)

deck = [Card(s,r) for s in suits for r in ranks]
shuffle_deck(deck)[:5]

[♠J, ♥Q, ♥A, ♠5, ♥10]

In [ ]:
#| export
def withdraw_card(deck: list[Card], n: int=1) -> list[Card]:
    "Withdraw `n` cards from `deck` and return them. Modifies `deck` in place."
    return [deck.pop() for _ in range(n)]

In [ ]:
deck = [Card(s,r) for s in suits for r in ranks]
withdraw_card(deck)

[♣A]

In [ ]:
assert len(deck) == 51

In [ ]:
withdraw_card(deck, n=5)
assert len(deck) == 46

## Player

`Player`class:

- balance: int
- bet: int
- playing: bool
- hand: List[Card] methods:
- take_action(round)
- call(round)
- check(round)
- fold(round)
- raise(round, amount).

In [ ]:
#| export
class Action(Enum):
    RAISE = auto()
    CALL = auto()
    CHECK = auto()
    FOLD = auto()

### Strategy

`Player` has a `strategy` attribute, which chooses actions.

In [ ]:
#| export
def human(player): 
    prompt = f"""This is {player.username}'s turn! Type an action: Call, Check, Fold, Raise X with X amount
     your current bet: {player.bet}, your hand: {player.hand}"""
    while True:
        act, *amnt = input(prompt).split()
        if (action := getattr(Action, act.upper(), None)):
            return (action, int(*amnt) if amnt else 0)
        print(f"Invalid action: {act}. Try again.")

In [ ]:
#| export
def always_call(player): return (Action.CALL, 0)

### player class

In [ ]:
#| export
@dataclass
class Player:
    username: str = 'new_user' # username
    balance: int = 0 # total balance
    bet: int = 0 # active bet this round
    playing: bool = True # playing this round or not
    hand: list = field(default_factory=list) # playing hand
    strategy: callable = always_call # a strategy the player uses to make actions (human or ai)

    def perform_action(self, stage_bet) -> tuple[Action, int, int]:
        ah = ActionHandler(self, stage_bet)
        return ah.handle()

In [ ]:
#| export
class ActionHandler:
    def __init__(self, player, stage_bet):
        self.player = player
        self.stage_bet = stage_bet
    
    def handle(self):
        action, amount = self.player.strategy(self.player)
        print(f'{self.player.username} intends to {action.name.lower()}, stage_bet: {self.stage_bet}, raise_amount: {amount}')
        return getattr(self, '_'+action.name.lower())(amount)

    def _call(self, amount):
        need = self.stage_bet - self.player.bet
        if self.player.balance < need:
            print(f"Cannot call. balance ({self.player.balance}) too low to match the stage_bet ({self.stage_bet}). Current bet: {self.player.bet}")
            return self.handle()
        else:
            self.player.balance -= need
            self.player.bet += need
            return need

    def _check(self, amount): return 0

    def _fold(self, amount):
        self.player.playing = False
        return 0

    def _raise(self, amount):
        new_stage_bet = self.stage_bet + amount
        need = new_stage_bet - self.player.bet
        if need > self.player.balance:
            print(f'Cannot raise by {amount}. Your balance: {self.player.balance}, current bet: {self.player.bet}, stage_bet: {self.stage_bet}')
            return self.handle()
        else:
            self.player.balance -= need
            self.player.bet += need
            return need

In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=always_call)
need = player1.perform_action(stage_bet)
need, player1

galopy intends to call, stage_bet: 2, raise_amount: 0


(2,
 Player(username='galopy', balance=18, bet=2, playing=True, hand=[], strategy=<function always_call>))

## Round

Round Management

Attributes:
- stage (pre-flop, flop, turn, river)
- players
- deck
- community_cards

Methods:

- `betting_round(round) -> None`: Manage one round of player actions (pre-flop, flop, turn, or river). Ends when all active players have called or checked.
- `distribute(pot, winners) -> None`: Distribute the pot to the winner(s).
- `start_round(round) -> None`: Orchestrate the whole hand: deal hole cards → blinds → betting round → flop → betting round → turn → betting round → river → betting round → showdown → distribute.

In [ ]:
#| export
class Stage(Enum):
    PREFLOP = auto()
    FLOP = auto()
    TURN = auto()
    RIVER = auto()

    def next(self):
        """Return the next stage, or None if already at RIVER."""
        return Stage(self.value + 1) if self.value < len(Stage) else None

Let's go through playing a round of poker game!

In [ ]:
#| export
class Round:
    def __init__(self, players = None, deck = None, community = None, stage = Stage.PREFLOP, stage_bet = 0, pot = 0):
        self.players = [] if players is None else players
        self.deck = shuffle_deck([Card(s,r) for s in suits for r in ranks]) if deck is None else deck
        self.community = [] if community is None else community
        self.stage = stage
        self.stage_bet = stage_bet
        self.pot = pot
    
    def state(self):
        "Return a string describing the current game situation."
        if self.stage is None: return "Game is over"
        player = self.players[self._idx]
        return f"{player.username}'s turn | Stage: {self.stage.name} | Hand: {player.hand} | \
        Community: {self.community} | Stage bet: {self.stage_bet} | Your bet: {player.bet} | Balance: {player.balance} | Pot: {self.pot}"

In [ ]:
#| export
@patch
def _distribute(self: Round, winners) -> None:
    "Distribute the prize pot to the winners"
    print(f"Distributing $ {self.pot} to {winners}")
    for player in winners:
        player.balance += self.pot // len(winners)

In [ ]:
#| export
@patch
def _prep_round(self: Round):
    "Prepare for the round by getting the deck, shuffle, deal hole hands."
    self.deck = shuffle_deck([Card(s,r) for s in suits for r in ranks])
    for p in self.players: 
        p.playing = True
        p.hand = withdraw_card(self.deck, 2)
    self.community = []
    self.pot = 0
    self.stage = Stage.PREFLOP

In [ ]:
#| export
@patch
def _reset_betting_round(self: Round):
    "Reset state for a new betting round."
    self._raiser, self.stage_bet, self._idx = -1, 0, 0
    for p in self.players: p.bet = 0

In [ ]:
#| export
@patch
def _deal_community(self: Round):
    "Deal community cards"
    if self.stage == Stage.FLOP: 
        self.community += withdraw_card(self.deck, 3)
    elif self.stage in (Stage.TURN, Stage.RIVER): 
        self.community += withdraw_card(self.deck)
    print(f"\n--- {self.stage.name} --- Community: {self.community}")

In [ ]:
#| export
@patch
def _get_next_player(self: Round) -> Player:
    while True:
        player = self.players[self._idx]
        if player.playing:
            if player.strategy is human:
                bet_amt = yield player
            else:
                bet_amt = player.perform_action(self.stage_bet)
            self.pot += bet_amt
            if player.bet > self.stage_bet:
                self._raiser = self._idx
                self.stage_bet = player.bet
        self._idx = (self._idx + 1) % len(self.players)
        if self._raiser == -1: self._raiser = 0
        if self._idx == self._raiser: break

In [ ]:
#| export
@patch
def _showdown(self: Round):
    "Evaluate and distribute."
    winning_score = compare_hands([p.hand + self.community for p in self.players if p.playing])
    winners = [p for p in self.players if evaluate_hand(p.hand + self.community) == winning_score]
    print(f"Winners: {[p.username for p in winners]} with {winning_score}")
    self._distribute(winners)

In [ ]:
#| export
@patch
def start_round(self: Round):
    "Deal, start PREFLOP, and auto-advance to first human player."
    self._prep_round()
    self._reset_betting_round()
    for s in Stage:
        self.stage = s
        self._deal_community()
        self._reset_betting_round()
        yield from self._get_next_player()
    self.stage = None
    self._showdown()

In [ ]:
#| export
def act(game, player, round):
    if player is None: print("Game is over"); return
    try:
        player = game.send(player.perform_action(round.stage_bet))
    except StopIteration:
        player = None
    return player

In [ ]:
player1 = Player('galopy', balance=20, playing=True)
player2 = Player('jalopy', balance=20, playing=True)
players = [player1, player2]
round1 = Round(players = players)
game = round1.start_round()
player = next(game, None)


--- PREFLOP --- Community: []


galopy intends to call, stage_bet: 0, raise_amount: 0


jalopy intends to call, stage_bet: 0, raise_amount: 0



--- FLOP --- Community: [♣A, ♣4, ♦7]


galopy intends to call, stage_bet: 0, raise_amount: 0


...


--- RIVER --- Community: [♣A, ♣4, ♦7, ♥A, ♣9]


galopy intends to call, stage_bet: 0, raise_amount: 0


jalopy intends to call, stage_bet: 0, raise_amount: 0


Winners: ['jalopy'] with (3, 14, 9, 8)


Distributing $ 0 to [Player(username='jalopy', balance=20, bet=0, playing=True, hand=[♣8, ♥9], strategy=<function always_call>)]


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

### Some notes at the end to consider:

I am not sure. Would Norvig use generator approach? I am not sure this approach is as clean as I thought it would have been. What would other approaches look like?

I am not sure. Would Norvig use generator approach? I am not sure this approach is as clean as I thought it would have been. What would other approaches look like?

I am not sure. Would Norvig use generator approach? I am not sure this approach is as clean as I thought it would have been. What would other approaches look like?

What would be nice approach if I eventually want to expand it to more players? And those players all have their actions on their separate computers. And everyone waits until it is their turn.